Copyright 2026 Google DeepMind Technologies Limited

All software is licensed under the Apache License, Version 2.0 (Apache 2.0);
you may not use this file except in compliance with the Apache 2.0 license.
You may obtain a copy of the Apache 2.0 license at:
https://www.apache.org/licenses/LICENSE-2.0

All other materials are licensed under the Creative Commons Attribution 4.0
International License (CC-BY). You may obtain a copy of the CC-BY license at:
https://creativecommons.org/licenses/by/4.0/legalcode

Unless required by applicable law or agreed to in writing, all software and
materials distributed here under the Apache 2.0 or CC-BY licenses are
distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND,
either express or implied. See the licenses for the specific language governing
permissions and limitations under those licenses.

This is not an official Google product.

In [ ]:
#@title Imports (run this first)

import math
import matplotlib.pyplot as plt
import numpy as np

# Packing unit regular hexagons inside a regular hexagon

Consider the problem of packing $n$ disjoint regular hexagons with unit side length into a larger regular hexagon, minimizing the side length of the outer
hexagon. The state of the art can be found on
[Erich Friedman's homepage](https://erich-friedman.github.io/packing/hexinhex/).

For $n=11$ and $n=12$, the best known constructions use outer hexagons of side
lengths $3.943$ and $4.0$, respectively. AlphaEvolve found packing arrangements
that improve these bounds to $3.931$ and $3.942$, respectively. These arrangements are shown below.

In [ ]:
#@title Verification code

# A regular hexagon on the plane is parameterized as a 4-tuple of floats:
# center_x: x-coordinate of the center.
# center_y: y-coordinate of the center.
# side_length: Length of each side.
# angle_degrees: Rotation angle in degrees (clockwise from horizontal).

def hexagon_vertices(
    center_x: float,
    center_y: float,
    side_length: float,
    angle_degrees: float,
) -> list[tuple[float, float]]:
  """Generates the vertices of a regular hexagon.

  Args:
    center_x: x-coordinate of the center.
    center_y: y-coordinate of the center.
    side_length: Length of each side.
    angle_degrees: Rotation angle in degrees (clockwise from horizontal).

  Returns:
    A list of tuples, where each tuple (x, y) represents the vertex location.
  """
  vertices = []
  angle_radians = math.radians(angle_degrees)
  for i in range(6):
    angle = angle_radians + 2 * math.pi * i / 6
    x = center_x + side_length * math.cos(angle)
    y = center_y + side_length * math.sin(angle)
    vertices.append((x, y))
  return vertices


def normalize_vector(v: tuple[float, float]) -> tuple[float, float]:
  """Normalizes a 2D vector to have unit norm."""
  magnitude = math.sqrt(v[0]**2 + v[1]**2)
  return (v[0] / magnitude, v[1] / magnitude) if magnitude != 0 else (0., 0.)


def get_normals(
    vertices: list[tuple[float, float]]
) -> list[tuple[float, float]]:
  """Returns the outward normals of a polygon's edges."""
  normals = []
  for i in range(len(vertices)):
    p1 = vertices[i]
    p2 = vertices[(i + 1) % len(vertices)]  # Wrap around to the first vertex.
    edge = (p2[0] - p1[0], p2[1] - p1[1])
    normal = normalize_vector((-edge[1], edge[0]))  # Rotate edge by 90 degrees.
    normals.append(normal)
  return normals


def project_polygon(
    vertices: list[tuple[float, float]],
    axis: tuple[float, float],
) -> tuple[float, float]:
  """Projects a polygon onto an axis and returns the min/max values."""
  min_proj = float('inf')
  max_proj = float('-inf')
  for vertex in vertices:
    projection = vertex[0] * axis[0] + vertex[1] * axis[1]  # Dot product.
    min_proj = min(min_proj, projection)
    max_proj = max(max_proj, projection)
  return min_proj, max_proj


def overlap_1d(min1: float, max1: float, min2: float, max2: float) -> bool:
  """Determines whether two 1D intervals overlap."""
  return max1 >= min2 and max2 >= min1


def polygons_intersect(
    vertices1: list[tuple[float, float]],
    vertices2: list[tuple[float, float]],
) -> bool:
  """Determines if two polygons intersect, using the Separating Axis Theorem."""
  normals1 = get_normals(vertices1)
  normals2 = get_normals(vertices2)
  axes = normals1 + normals2

  for axis in axes:
    min1, max1 = project_polygon(vertices1, axis)
    min2, max2 = project_polygon(vertices2, axis)
    if not overlap_1d(min1, max1, min2, max2):
      return False  # Separating axis found, polygons are disjoint.
  return True  # No separating axis found, polygons intersect.


def hexagons_are_disjoint(
    hex1_params: tuple[float, float, float, float],
    hex2_params: tuple[float, float, float, float],
) -> bool:
  """Determines if two hexagons have disjoint interiors, given their parameters."""
  hex1_vertices = hexagon_vertices(*hex1_params)
  hex2_vertices = hexagon_vertices(*hex2_params)
  return not polygons_intersect(hex1_vertices, hex2_vertices)


def is_inside_hexagon(
    point: tuple[float, float],
    hex_params: tuple[float, float, float, float],
) -> bool:
  """Checks if a point is inside a hexagon, given its parameters."""
  hex_vertices = hexagon_vertices(*hex_params)
  for i in range(len(hex_vertices)):
    p1 = hex_vertices[i]
    p2 = hex_vertices[(i + 1) % len(hex_vertices)]
    edge_vector = (p2[0] - p1[0], p2[1] - p1[1])
    point_vector = (point[0] - p1[0], point[1] - p1[1])
    cross_product = (
        edge_vector[0] * point_vector[1] - edge_vector[1] * point_vector[0]
    )
    if cross_product < 0:
      return False
  return True


def all_hexagons_contained(
    inner_hex_params_list: list[tuple[float, float, float, float]],
    outer_hex_params: tuple[float, float, float, float],
) -> bool:
  """Checks if all inner hexagons are contained within the outer hexagon."""
  for inner_hex_params in inner_hex_params_list:
    inner_hex_vertices = hexagon_vertices(*inner_hex_params)
    for vertex in inner_hex_vertices:
      if not is_inside_hexagon(vertex, outer_hex_params):
        return False
  return True


def verify_construction(
    inner_hex_data: tuple[float, float, float],
    outer_hex_center: tuple[float, float],
    outer_hex_side_length: float,
    outer_hex_angle_degrees: float,
):
  """Verifies the hexagon packing construction with a rotated outer hexagon.

  Args:
    inner_hex_data: List of (x, y, angle_degrees) tuples for inner hexagons.
      (The inner hexagons have unit side lengths.)
    outer_hex_center: (x, y) tuple for the outer hexagon center.
    outer_hex_side_length: Side length of the outer hexagon.
    outer_hex_angle_degrees: Rotation angle of the outer hexagon in degrees.

  Raises:
    AssertionError if the construction is not valid.
  """

  inner_hex_params_list = [
      (x, y, 1, angle) for x, y, angle in inner_hex_data
  ]  # Sets the side length to 1.
  outer_hex_params = (
      outer_hex_center[0], outer_hex_center[1],
      outer_hex_side_length, outer_hex_angle_degrees
  )

  # Disjointness check.
  for i in range(len(inner_hex_params_list)):
    for j in range(i + 1, len(inner_hex_params_list)):
      if not hexagons_are_disjoint(
          inner_hex_params_list[i], inner_hex_params_list[j]
      ):
        raise AssertionError(f"Hexagons {i+1} and {j+1} intersect!")

  # Containment check.
  if not all_hexagons_contained(inner_hex_params_list, outer_hex_params):
    raise AssertionError(
        "Not all inner hexagons are contained in the outer hexagon!"
    )

  print("Verified that this is a valid set of disjoint unit hexagons contained in the outer hexagon.")

In [ ]:
#@title Plotting code

def plot_construction(
    inner_hex_data: list[tuple[float, float, float]],
    outer_hex_center: tuple[float, float],
    outer_hex_side_length: float,
    outer_hex_angle_degrees: float,
):
  """Plots the hexagon packing construction using matplotlib.

  Args:
    inner_hex_data: List of (x, y, angle_degrees) tuples for inner hexagons.
    outer_hex_center: (x, y) tuple for the outer hexagon center.
    outer_hex_side_length: Side length of the outer hexagon.
    outer_hex_angle_degrees: Rotation angle of the outer hexagon in degrees.
  """

  inner_hex_params_list = [(x, y, 1, angle) for x, y, angle in inner_hex_data]
  outer_hex_params = (
      outer_hex_center[0], outer_hex_center[1],
      outer_hex_side_length, outer_hex_angle_degrees
  )

  _, ax = plt.subplots(figsize=(6, 6))
  ax.set_aspect('equal')

  # Plot outer hexagon (in red).
  outer_hex_vertices = hexagon_vertices(*outer_hex_params)
  outer_hex_x, outer_hex_y = zip(*outer_hex_vertices)
  ax.plot(
      outer_hex_x + (outer_hex_x[0],),
      outer_hex_y + (outer_hex_y[0],),
      'r-',
      label='Outer Hexagon',
  )

  # Plot inner hexagons (in blue).
  for i, inner_hex_params in enumerate(inner_hex_params_list):
    inner_hex_vertices = hexagon_vertices(*inner_hex_params)
    inner_hex_x, inner_hex_y = zip(*inner_hex_vertices)
    ax.fill(
        inner_hex_x + (inner_hex_x[0],),
        inner_hex_y + (inner_hex_y[0],),
        alpha=0.7,
        color='blue',
        label='Inner Hexagons' if i == 0 else None,  # Label only once.
    )

  ax.set_title('Hexagon packing construction discovered by AlphaEvolve')
  ax.legend()
  ax.grid(False)
  plt.show()

In [ ]:
#@title Construction 1 (11 hexagons): Data

inner_hex_data = [
    [7.227468518248726, 1.406189555916512, 64.96835531696001],
    [9.108934177226912, 6.282338181661271, 82.7839419155727],
    [9.953071339809288, 4.757920599499527, 22.75943751294596],
    [9.119104027817858, 1.3635244317245632, 64.98416780884727],
    [8.202877059101247, 4.72907294863739, 22.702381798755646],
    [7.272127593730819, 3.260296487997907, 202.68420631619662],
    [9.055201371937533, 3.2081226414835964, 82.70304387527644],
    [11.605104986630607, 3.9103231530709373, 120.97244798016398],
    [10.63107376404371, 2.224090366073967, 244.99786687183936],
    [6.427221005557855, 4.784445729508918, 262.6855070070916],
    [7.357815645136251, 6.25309652428302, 22.7012724369423],
]

print("Number of inner hexagons:", len(inner_hex_data))
outer_hex_center = [8.675544349693055, 3.861025611329817]
outer_hex_side_length = 3.930092
outer_hex_angle_degrees = 60.96370590710646

In [ ]:
#@title Construction 1 (11 hexagons): Verification and plotting

verify_construction(
    inner_hex_data, outer_hex_center,
    outer_hex_side_length, outer_hex_angle_degrees
)
print(f"Outer hexagon side length: {repr(float(outer_hex_side_length))}.")
plot_construction(
    inner_hex_data, outer_hex_center,
    outer_hex_side_length, outer_hex_angle_degrees
)

In [ ]:
#@title Construction 2 (12 hexagons): Data

inner_hex_data = [
    [-4.189650776376798, 1.8191996226926388, 27.083656548467786],
    [-4.1762891342159225, 7.064090887028092, 267.1521094125118],
    [-2.742942219672671, 3.9055564132509595, 27.098852933212644],
    [-6.194948523724643, 4.243623380441444, 87.15308054130165],
    [-2.4735773559547236, 6.43056712077689, 27.144807633484653],
    [-3.8735033091694633, 5.272827562035259, 207.12012595575163],
    [-1.040517842157661, 3.2724339122261963, 147.11193087313353],
    [-1.3431542020437193, 5.063486444235776, 27.124004797633685],
    [-5.89197412464008, 2.452466762507662, 207.10748153590382],
    [-4.492206360794807, 3.610020660157929, 327.09918513988094],
    [-2.4402287984241076, 2.1146830723057635, 147.1073847274004],
    [-5.576327284388157, 5.906127413424367, 207.16788311720808],
]

print("Number of inner hexagons:", len(inner_hex_data))
outer_hex_center = [-3.7029107170331157, 4.262832492467158]
outer_hex_side_length = 3.9419123
outer_hex_angle_degrees = 219.59113156095745

In [ ]:
#@title Construction 2 (12 hexagons): Verification and plotting

verify_construction(
    inner_hex_data, outer_hex_center,
    outer_hex_side_length, outer_hex_angle_degrees
)
print(f"Outer hexagon side length: {repr(float(outer_hex_side_length))}.")
plot_construction(
    inner_hex_data, outer_hex_center,
    outer_hex_side_length, outer_hex_angle_degrees
)